# Notebook 03 — Automatic Clustering
## Best Case vs. Worst Case

**Key Insight**: When your queries don't align with natural load order, a cluster key reorganizes partitions to match your access pattern.

| | Best Design | Worst Design |
|---|---|---|
| **Cluster key** | `account_id` (matches customer lookup pattern) | `currency_code` (only 3 values — useless) |
| **After reclustering** | 95%+ partition pruning | Zero improvement, wasted credits |

In [ ]:
USE ROLE SYSADMIN;
USE WAREHOUSE HOL_XS;
USE SCHEMA JPMC_PERF_HOL.CONSUMER_BANKING;

---
## Step 1: Baseline — The Problem (from Notebook 02)

Customer account lookups scan nearly every partition because `account_id` is randomly distributed.

In [ ]:
-- Baseline: customer lookup WITHOUT clustering (slow — full scan)
SELECT transaction_date, merchant_name, amount, channel
FROM TRANSACTIONS
WHERE account_id = 'ACC-0000012345'
ORDER BY transaction_date DESC
LIMIT 50;

**Note the Query Profile**: partitions scanned ≈ total partitions

---
## Step 2: Best Case — Add Cluster Key on `account_id`

We'll create a clustered copy to compare immediately (reclustering the 500M-row table in-place takes time).

In [ ]:
-- Create a clustered copy (ordered by account_id during CTAS)
CREATE OR REPLACE TABLE TRANSACTIONS_CLUSTERED
    CLUSTER BY (account_id)
AS
SELECT * FROM TRANSACTIONS
ORDER BY account_id;

In [ ]:
-- Verify clustering quality
SELECT SYSTEM$CLUSTERING_INFORMATION('TRANSACTIONS_CLUSTERED', '(account_id)');

In [ ]:
-- BEST CASE: Same query, now on clustered table
SELECT transaction_date, merchant_name, amount, channel
FROM TRANSACTIONS_CLUSTERED
WHERE account_id = 'ACC-0000012345'
ORDER BY transaction_date DESC
LIMIT 50;

**Open the Query Profile** → Compare:
- Original table: ~90-100% partitions scanned
- Clustered table: <5% partitions scanned
- Execution time: dramatic improvement

---
## Step 3: Worst Case — Clustering on a Useless Column

What happens if you cluster on `currency_code`? (only 3 distinct values: USD, GBP, EUR)

In [ ]:
-- Check: currency_code is ALREADY perfectly clustered (low cardinality = few values per partition anyway)
SELECT SYSTEM$CLUSTERING_INFORMATION('TRANSACTIONS', '(currency_code)');

In [ ]:
-- Estimate the cost of clustering on currency_code — money for nothing
SELECT SYSTEM$ESTIMATE_AUTOMATIC_CLUSTERING_COSTS('TRANSACTIONS', '(currency_code)');

**What you'll see**: The table is already well-clustered on `currency_code` because with only 3 values, most partitions naturally contain a limited range. Adding a cluster key here burns credits for zero improvement.

---
## Step 4: Side-by-Side Metrics

In [ ]:
-- Compare the two account_id queries
SELECT
    CASE WHEN query_text ILIKE '%TRANSACTIONS_CLUSTERED%' THEN 'CLUSTERED' ELSE 'UNCLUSTERED' END AS version,
    total_elapsed_time / 1000 AS elapsed_sec,
    query_id,
    bytes_scanned / (1024*1024*1024) AS gb_scanned
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(minute, -15, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 20
))
WHERE start_time > DATEADD(minute, -15, CURRENT_TIMESTAMP())
WHERE query_text ILIKE '%ACC-0000012345%'
ORDER BY start_time DESC
LIMIT 2;

---
## Step 5: Cost Estimation

In [ ]:
-- What would it cost to maintain clustering on account_id for the original table?
SELECT SYSTEM$ESTIMATE_AUTOMATIC_CLUSTERING_COSTS('TRANSACTIONS', '(account_id)');

---
## Key Takeaways

| | Best Design | Worst Design |
|---|---|---|
| **Cluster key** | Column you frequently filter on with high cardinality (`account_id`) | Column with 3 values (`currency_code`) |
| **Result** | 95%+ pruning, 10-20x faster | Zero improvement, wasted reclustering credits |
| **When to use** | Queries don't align with natural load order | NEVER cluster on low-cardinality columns |

**Decision framework**:
1. Check natural clustering first (`SYSTEM$CLUSTERING_INFORMATION`)
2. If already good → don't add a cluster key (free performance)
3. If poor AND you query that column frequently → add cluster key
4. Estimate cost before committing (`SYSTEM$ESTIMATE_AUTOMATIC_CLUSTERING_COSTS`)

**Next** → [Notebook 04 — Warehouse Optimization](./Notebook_04_Warehouse_Optimization.ipynb)

---
## Cleanup (optional)

In [ ]:
-- Keep TRANSACTIONS_CLUSTERED for later notebooks if desired
-- DROP TABLE IF EXISTS TRANSACTIONS_CLUSTERED;